# Model Compare · qwen2.5vl vs qwen3.5
Run both models on the same photo with the same structured prompt, then compare side-by-side.

## Config & Imports

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

# ── config ────────────────────────────────────────────────────
PROCESS_PHOTO = "Kinan.Sweidan-22.JPG"  # set to None for all photos
MODEL_1 = "qwen2.5vl:latest"            # vision model — image is passed directly
MODEL_2 = "qwen3.5:9b"                  # text model — thinking disabled

# ── imports ───────────────────────────────────────────────────
from pathlib import Path
from PIL import Image
import base64, urllib.request, json, matplotlib.pyplot as plt, time
from IPython.display import display as ipy_display, HTML

RAW_DIR = Path("./data/raw")
photos = [RAW_DIR / PROCESS_PHOTO] if PROCESS_PHOTO else sorted(
    p for ext in ("*.jpg", "*.JPG") for p in RAW_DIR.glob(ext)
)

# ── helpers ───────────────────────────────────────────────────
def ollama_vision(prompt, image_b64, model=MODEL_1):
    """Call a vision model via Ollama /api/generate with an image."""
    payload = {"model": model, "prompt": prompt, "stream": False, "images": [image_b64]}
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.loads(r.read())["response"]

def ollama_text(prompt, model=MODEL_2):
    """Call a text model via Ollama /api/generate with thinking disabled."""
    payload = {"model": model, "prompt": prompt, "stream": False, "think": False}
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.loads(r.read())["response"]

results = {}  # shared state across cells
print(f"Photos to process: {[p.name for p in photos]}")

## Prompt

In [ ]:
ANALYSIS_PROMPT = """You are analyzing a black and white street photograph by Chicago photographer Kinan Sweidan.

Describe each field below. Be specific and concise.

subject: [primary subject, position, action — 1-2 sentences]
composition: [framing, rule of thirds, leading lines, negative space — 1-2 sentences]
lighting: [direction, quality, contrast, shadow detail — 1-2 sentences]
mood: [emotional tone, atmosphere — 1 sentence]
technical: [depth of field, motion blur, grain if visible — 1 sentence]
style_notes: [distinctive visual choices specific to street or portrait photography — 1 sentence]

Respond with only the key-value pairs above. No reasoning. No commentary. Nothing else."""

print(ANALYSIS_PROMPT)

## Model 1 · qwen2.5vl (Vision)

In [ ]:
for path in photos:
    with open(path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()

    t0 = time.time()
    m1_out = ollama_vision(ANALYSIS_PROMPT, img_b64, model=MODEL_1)
    t1 = time.time()

    results[path] = {"path": path, "img_b64": img_b64, "m1": m1_out, "t1": t1 - t0}
    print(f"✓ {path.name}  ({t1-t0:.1f}s)  [{MODEL_1}]\n\n{m1_out}\n")

## Model 2 · qwen3.5:9b (Text, thinking off)
The image description from Model 1 is passed as context so both models analyze the same photograph.

In [ ]:
for path, r in results.items():
    # Prepend Model 1 description so qwen3.5 has visual context
    context_prompt = (
        f"Visual description of the photograph (from a vision model):\n{r['m1']}\n\n"
        + ANALYSIS_PROMPT
    )

    t0 = time.time()
    m2_out = ollama_text(context_prompt, model=MODEL_2)
    t2 = time.time()

    r["m2"] = m2_out
    r["t2"] = t2 - t0
    print(f"✓ {path.name}  ({t2-t0:.1f}s)  [{MODEL_2}]\n\n{m2_out}\n")

## Results · Side-by-Side

In [ ]:
for path, r in results.items():
    fig, ax = plt.subplots(figsize=(4, 5))
    ax.imshow(Image.open(path))
    ax.axis("off")
    ax.set_title(path.name, fontsize=9, color="#888")
    plt.tight_layout()
    plt.show()

    t1 = r.get("t1", 0)
    t2 = r.get("t2", 0)

    ipy_display(HTML(f"""
    <div style="display:grid; grid-template-columns:1fr 1fr; gap:24px; font-family:monospace; font-size:12px; max-width:1200px; margin-top:8px">
      <div>
        <div style="color:#888; font-size:10px; margin-bottom:6px; border-bottom:1px solid #eee; padding-bottom:4px">
          MODEL 1 &nbsp;&middot;&nbsp; {MODEL_1} &nbsp;&middot;&nbsp; {t1:.1f}s
        </div>
        <div style="white-space:pre-wrap; line-height:1.8">{r.get('m1', '— run Model 1 cell first')}</div>
      </div>
      <div>
        <div style="color:#888; font-size:10px; margin-bottom:6px; border-bottom:1px solid #eee; padding-bottom:4px">
          MODEL 2 &nbsp;&middot;&nbsp; {MODEL_2} &nbsp;&middot;&nbsp; {t2:.1f}s &nbsp;&middot;&nbsp; thinking: off
        </div>
        <div style="white-space:pre-wrap; line-height:1.8">{r.get('m2', '— run Model 2 cell first')}</div>
      </div>
    </div>
    <div style="color:#aaa; font-size:10px; margin-top:10px">
      Total: {t1+t2:.1f}s &nbsp;|&nbsp; {MODEL_1}: {t1:.1f}s &nbsp;|&nbsp; {MODEL_2}: {t2:.1f}s
    </div>
    <hr style="margin:20px 0; border:none; border-top:1px solid #eee">
    """))